# NB10 — Cold-Paper Explanation (Interview Preparation)

**Module:** 19 — Research Seminar and Paper Reading  
**Estimated time:** 2 hours + ongoing practice  

---

## Motivation

This is the most direct interview-prep notebook in the programme. The task is specific: a bioinformatics/computational biology interviewer hands you an abstract, gives you 5 minutes to read it, and asks "can you walk me through this?"

The skill being tested is not memorization of facts. It is the ability to rapidly extract the structure of a scientific argument from unfamiliar text and communicate it clearly. Pass-3 practice (NB01) builds this skill over months. This notebook operationalizes the interview format.

## 1. The Three-Step Cold Explanation

After 5 minutes of reading an abstract, structure your explanation in exactly this order:

**Step 1 — Problem (30 seconds):**  
"This paper is addressing the problem of [X]. Before this work, the challenge was [Y]."

**Step 2 — Approach (60–90 seconds):**  
"What they did was [method at a high level — what input, what transformation, what output, and why that approach makes sense for this problem]."

**Step 3 — Finding (30–60 seconds):**  
"The key result is [finding], which matters because [implication]."

Then, **one honest observation:**  
"One thing I would want to check is [limitation or question] — because [reason]."

Total: 2.5–3 minutes. Leave 2 minutes for the interviewer's follow-up.

## 2. Adjusting for Audience

| Audience | Adjust this | Keep this |
|----------|------------|----------|
| Biologist | Reduce algorithmic jargon; emphasize biological implication | The "what" and "why" |
| Computer scientist | Reduce biological jargon; emphasize data structure and algorithm | The "how" |
| General scientist | No jargon; use analogy for the method | The problem and the key result |

## 3. Difficulty Levels

| Level | Abstract type | Skill tested |
|-------|-------------|-------------|
| 1 | Short methods paper with clear structure | Basic extraction |
| 2 | Multi-experiment paper with multiple claims | Claim prioritization |
| 3 | Theory/math-heavy paper | Translating formalism to plain English |
| 4 | Preprint with unclear contribution | Critical evaluation under uncertainty |

In [ ]:
import time
import re
from dataclasses import dataclass, field
from typing import List, Optional, Dict
import numpy as np
import matplotlib.pyplot as plt


# Five real abstracts from the programme's papers.md, with difficulty ratings
PRACTICE_ABSTRACTS = [
    {
        'cite': 'Needleman & Wunsch (1970)',
        'difficulty': 1,
        'abstract': (
            "A computer adaptable method for finding similarities in the amino acid sequences "
            "of two proteins has been developed. From these findings it is possible to determine "
            "which of the proteins share a maximum homology. The method to determine the best fit "
            "uses a matrix of score values generated from a comparison of all possible pairs of "
            "amino acids in the two sequences. A 'goodness of fit' value is calculated for each "
            "comparison and accumulated in a matrix. Backtracking from the highest accumulated "
            "score allows the best alignment of the amino acid sequences to be found."
        ),
        'key_terms': ['alignment', 'matrix', 'amino acid', 'homology', 'backtracking', 'score'],
        'model_problem': 'How do you find the most similar regions between two protein sequences?',
        'model_approach': 'Build a matrix of pairwise amino acid comparison scores; accumulate them along diagonal paths; backtrack from the highest score to find the best alignment.',
        'model_finding': 'The dynamic programming approach finds the globally optimal alignment between two sequences.',
    },
    {
        'cite': 'Love et al. (2014) — DESeq2',
        'difficulty': 2,
        'abstract': (
            "In comparative high-throughput sequencing assays, a fundamental task is the "
            "analysis of count data, such as read counts per gene in RNA-seq, for evidence of "
            "systematic changes across experimental conditions. Small replicate numbers, "
            "discreteness, large dynamic range and the presence of outliers require a "
            "combination of statistical methods and domain knowledge. We present DESeq2, "
            "a method for differential analysis of count data, using shrinkage estimation "
            "for dispersions and fold changes to improve stability and interpretability of "
            "estimates. This enables a more quantitative analysis focused on the "
            "effect size and its uncertainty."
        ),
        'key_terms': ['DESeq2', 'RNA-seq', 'count data', 'shrinkage', 'fold change', 'dispersion', 'differential expression'],
        'model_problem': 'How do you detect genes that change in expression between conditions when sample sizes are small?',
        'model_approach': 'Model read counts as negative binomial; use shrinkage (empirical Bayes) to stabilize dispersion estimates across genes with small replicate numbers.',
        'model_finding': 'Shrinkage-estimated fold changes are more stable than raw fold changes, particularly for low-count genes.',
    },
    {
        'cite': 'Keshav (2007) — How to Read a Paper',
        'difficulty': 1,
        'abstract': (
            "Researchers spend a great deal of time reading research papers. However, this skill "
            "is rarely taught, leading to much wasted effort. This article outlines a practical "
            "and efficient three-pass method for reading research papers. I also describe how "
            "to use this method to do a literature survey."
        ),
        'key_terms': ['three-pass', 'reading', 'literature survey', 'research papers', 'efficient'],
        'model_problem': 'Researchers waste time reading papers inefficiently because no one teaches them how.',
        'model_approach': 'A three-pass method: skim (structure), read (intent), reconstruct (mastery).',
        'model_finding': 'The three-pass method saves time and improves comprehension of complex papers.',
    },
    {
        'cite': 'Ioannidis (2005) — Why Most Findings Are False',
        'difficulty': 2,
        'abstract': (
            "There is increasing concern that most current published research findings are false. "
            "The probability that a research claim is true may depend on study power and bias, "
            "the number of other studies on the same question, and, importantly, the ratio of "
            "true to no relationships among the relationships probed in each scientific field. "
            "In this framework, a research finding is less likely to be true when the studies "
            "conducted in a field are small; when effect sizes are small; when there is a "
            "greater number and lesser preselection of tested relationships; when there is "
            "greater flexibility in designs, definitions, outcomes, and analytical modes; "
            "when there is greater financial and other interest and prejudice; and when more "
            "teams are involved in a scientific field in chase of statistical significance."
        ),
        'key_terms': ['false discoveries', 'statistical power', 'effect size', 'bias', 'p-hacking', 'replication'],
        'model_problem': 'Why are so many published scientific findings not reproduced?',
        'model_approach': 'Bayesian framework: model the probability a claim is true given study power, bias, and the prior probability of a true relationship in the field.',
        'model_finding': 'Most published findings are false, especially in low-powered, high-multiplicity studies.',
    },
    {
        'cite': 'McInnes et al. (2018) — UMAP',
        'difficulty': 3,
        'abstract': (
            "UMAP (Uniform Manifold Approximation and Projection) is a novel manifold learning "
            "technique for dimension reduction. UMAP is constructed from a theoretical framework "
            "based in Riemannian geometry and algebraic topology. The result is a practical "
            "scalable algorithm that applies to real world data. The UMAP algorithm is competitive "
            "with t-SNE for visualization quality, and arguably preserves more of the global "
            "structure with superior run time performance. Furthermore, UMAP has no computational "
            "restrictions on embedding dimension, making it viable as a general purpose dimension "
            "reduction technique for machine learning."
        ),
        'key_terms': ['UMAP', 'manifold learning', 'dimension reduction', 'Riemannian geometry', 'algebraic topology', 't-SNE', 'embedding'],
        'model_problem': 'How do you visualize high-dimensional data (like scRNA-seq expression profiles) in 2D while preserving structure?',
        'model_approach': 'Approximate a topological structure of the data manifold using fuzzy simplicial sets, then optimize a low-dimensional embedding to match it.',
        'model_finding': 'UMAP is faster than t-SNE and better preserves global structure, while producing comparable local clustering.',
    },
]


@dataclass
class ExplanationAttempt:
    abstract_cite: str
    explanation_text: str
    time_seconds: float
    key_terms: List[str]

    def score(self) -> Dict[str, float]:
        """Score the explanation on 5 criteria (0-2 each = 10 total)."""
        text_lower = self.explanation_text.lower()

        # 1. Problem statement present
        problem_signals = ['problem', 'challenge', 'question', 'task', 'difficulty', 'issue']
        problem_score = 2.0 if any(s in text_lower for s in problem_signals) else 0.0

        # 2. Approach described
        approach_signals = ['method', 'approach', 'algorithm', 'technique', 'model', 'use', 'apply']
        approach_score = 2.0 if any(s in text_lower for s in approach_signals) else 0.0

        # 3. Finding stated
        finding_signals = ['result', 'found', 'show', 'demonstrate', 'achieve', 'improve', 'outperform']
        finding_score = 2.0 if any(s in text_lower for s in finding_signals) else 0.0

        # 4. Key terms coverage
        term_coverage = sum(1 for t in self.key_terms if t.lower() in text_lower)
        term_score = 2.0 * (term_coverage / max(len(self.key_terms), 1))

        # 5. Length appropriateness (100-400 words)
        n_words = len(self.explanation_text.split())
        if 80 <= n_words <= 400:
            length_score = 2.0
        elif 50 <= n_words < 80 or 400 < n_words <= 600:
            length_score = 1.0
        else:
            length_score = 0.0

        return {
            'problem': problem_score,
            'approach': approach_score,
            'finding': finding_score,
            'key_terms': round(term_score, 1),
            'length': length_score,
            'total': round(problem_score + approach_score + finding_score + term_score + length_score, 1),
        }


class ColdExplanationTimer:
    """Simulate a timed cold-explanation exercise with scoring."""

    def __init__(self, paper_db: List[dict]):
        self.paper_db = paper_db
        self.attempts: List[ExplanationAttempt] = []

    def get_abstract(self, index: int) -> dict:
        return self.paper_db[index % len(self.paper_db)]

    def submit_explanation(self, paper_cite: str, explanation_text: str,
                           time_seconds: float, key_terms: List[str]) -> Dict:
        attempt = ExplanationAttempt(paper_cite, explanation_text, time_seconds, key_terms)
        self.attempts.append(attempt)
        return attempt.score()

    def score_history(self) -> List[float]:
        return [a.score()['total'] for a in self.attempts]


# Demonstrate with a sample explanation for the Needleman-Wunsch abstract
timer = ColdExplanationTimer(PRACTICE_ABSTRACTS)
paper = timer.get_abstract(0)
print(f"Practice abstract: {paper['cite']} (Difficulty: {paper['difficulty']}/4)")
print(f"\nAbstract:\n{paper['abstract']}")

sample_explanation = """
This paper addresses the problem of comparing two protein sequences to find how similar
they are. Before this work, there was no systematic computational method for this task.

The approach they used is dynamic programming. They build a matrix where each cell
scores the similarity between one amino acid from each sequence. The algorithm accumulates
these scores along paths through the matrix, then uses backtracking to find the path
with the highest total score — this path is the best alignment.

The key result is that this method finds the globally optimal alignment between any two
sequences. This matters because it laid the foundation for all modern sequence alignment
tools, including BLAST and modern genome aligners.

One thing I would want to check: the method runs in O(n*m) time where n and m are
sequence lengths. For genome-scale sequences (millions of bases), this is too slow —
which is why BLAST was developed as a heuristic alternative.
"""

scores = timer.submit_explanation(
    paper_cite=paper['cite'],
    explanation_text=sample_explanation,
    time_seconds=180,
    key_terms=paper['key_terms']
)
print(f"\nExplanation scores:")
for criterion, score in scores.items():
    print(f"  {criterion:12s}: {score}/2" if criterion != 'total' else f"  {'TOTAL':12s}: {score}/10")

In [ ]:
# Simulate a 10-week practice log showing improvement
np.random.seed(42)
simulated_scores = []
for week in range(10):
    base_score = 4 + week * 0.5 + np.random.normal(0, 0.5)
    simulated_scores.append(min(max(base_score, 0), 10))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Cold-Paper Explanation — Practice and Scoring', fontsize=12, fontweight='bold')

# Panel 1: Score breakdown radar
ax1 = fig.add_subplot(131, polar=True)
criteria = ['Problem', 'Approach', 'Finding', 'Key Terms', 'Length']
score_vals = [scores[k.lower().replace(' ', '_')] if k.lower().replace(' ', '_') in scores
              else scores.get(k.lower().split()[0], 0) for k in criteria]
score_vals_mapped = [
    scores['problem'], scores['approach'], scores['finding'],
    scores['key_terms'], scores['length']
]
max_per = 2.0
radar_vals = [v / max_per for v in score_vals_mapped]
N = len(criteria)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]
radar_vals += radar_vals[:1]
ax1.plot(angles, radar_vals, 'o-', linewidth=2, color='steelblue')
ax1.fill(angles, radar_vals, alpha=0.2, color='steelblue')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(criteria, size=9)
ax1.set_ylim(0, 1)
ax1.set_title('Explanation Quality\nScorecard', pad=20)

# Panel 2: Practice score over time
ax2 = axes[1]
weeks = list(range(1, len(simulated_scores) + 1))
ax2.plot(weeks, simulated_scores, 'o-', color='steelblue', linewidth=2, markersize=6)
ax2.axhline(7.0, color='green', linestyle='--', alpha=0.7, label='Target (7/10)')
z = np.polyfit(weeks, simulated_scores, 1)
p = np.poly1d(z)
ax2.plot(weeks, p(weeks), '--', color='orange', alpha=0.7, label='Trend')
ax2.set_xlabel('Practice session (week)')
ax2.set_ylabel('Total score (0–10)')
ax2.set_title('Cold Explanation Score\nOver Practice Sessions')
ax2.set_ylim(0, 10)
ax2.legend()
ax2.grid(True, alpha=0.3)

# Panel 3: Jargon density model
ax3 = axes[2]
jargon_levels = np.linspace(0, 10, 100)
audience_bio = 10 - jargon_levels  # biologist: prefers low jargon
audience_cs = 5 + jargon_levels * 0.4  # computer scientist: moderate jargon ok
audience_gen = np.maximum(10 - 1.5 * jargon_levels, 0)  # general: lower is better
ax3.plot(jargon_levels, audience_bio, color='#4CAF50', label='Biologist audience', linewidth=2)
ax3.plot(jargon_levels, audience_cs, color='#2196F3', label='CS audience', linewidth=2)
ax3.plot(jargon_levels, audience_gen, color='#FF9800', label='General audience', linewidth=2)
ax3.set_xlabel('Jargon density (0=plain, 10=technical)')
ax3.set_ylabel('Audience comprehension score')
ax3.set_title('Jargon Level vs Audience\nComprehension')
ax3.legend(fontsize=9)
ax3.set_ylim(0, 12)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cold_explanation.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercises

**Exercise 1:** Read the DESeq2 abstract (abstract index 1 in `PRACTICE_ABSTRACTS`) for exactly 5 minutes. Then close it and write a cold explanation in 150–300 words. Submit to `ColdExplanationTimer.submit_explanation()`. What is your score?

**Exercise 2:** Explain the UMAP abstract to a biologist (no math, one analogy for the manifold concept). Then explain it to a computer scientist (algorithmic detail). Score both. Which is harder?

**Exercise 3:** Find an abstract from a paper you have never read before (from any current module's papers.md). Apply the three-step structure. Time yourself.

**Exercise 4 (reflection):** After attempting Exercise 1, what did you miss in the DESeq2 explanation? What would you add in a second attempt?

## Quiz

1. What are the three steps of a cold explanation?
2. What is the "one honest observation" and why does it demonstrate scientific maturity?
3. For a biologist audience, which element of the three-step explanation gets the most time?
4. Why is the three-step structure more effective than starting with the methods?
5. If the interviewer asks "what statistical test do they use?" and you don't know, what is the correct response?

## References

- PRACTICE_ABSTRACTS in this notebook — use them for weekly practice
- paper-notes/ at the repo root — every logged paper is a practice abstract